In [ ]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie_short.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

1.

In [ ]:
data_gdynia["number_of_storeys"].agg(["min", "max", "mean", "median"])

In [ ]:
data_gdynia["number_of_storeys"].quantile([0.25, 0.75, 0.9])

In [ ]:
data_gdynia["number_of_storeys"].value_counts()[:1]

In [ ]:
(data_gdynia["number_of_storeys"] >= 5).mean()

In [ ]:
data_gdynia.loc[data_gdynia["number_of_storeys"].idxmax()]

In [ ]:
assert np.isclose(
    data_gdynia["number_of_storeys"].agg(["min", "max", "mean", "median"]),
    [1, 38, 3.79, 4],
    atol=0.01,
).all()
assert (data_gdynia["number_of_storeys"].quantile([0.25, 0.75, 0.9]) == [1, 6, 7]).all()

assert (
    data_gdynia["number_of_storeys"]
    .value_counts()
    .sort_values(ascending=False)[:1]
    .index
    == 1
)
assert (
    data_gdynia["number_of_storeys"].value_counts().sort_values(ascending=False)[:1]
    == 379
).all()

assert np.isclose((data_gdynia["number_of_storeys"] >= 5).mean(), 0.4796, atol=0.01)

assert data_gdynia["number_of_storeys"].idxmax() == 257
assert (
    data_gdynia.loc[
        data_gdynia["number_of_storeys"].idxmax(), ["addr_street", "addr_number"]
    ]
    == ["Antoniego Hryniewickiego", "6"]
).all()

2.

In [ ]:
memory_diff = (
    pd.concat(
        {
            "before": data_gdynia.memory_usage(deep=True),
            "after": data_gdynia.astype(
                {
                    "addr_street": "category",
                    "source": "category",
                    "arch_style": "category",
                    "upper_floor_function": "category",
                    "ground_floor_function": "category",
                    "osm_id": "category",
                }
            ).memory_usage(deep=True),
        },
        axis=1,
    )
    .drop("Index")
    .assign(ratio=lambda d: d["before"] / d["after"])
    .assign(nunique=lambda d: data_gdynia.nunique())
    .sort_values("ratio", ascending=False)
)

memory_diff

When all elements are unique saving as categories doesnt have sense, becouse it takes more memory. The smaller nunique the less memory the series takes.

3.

In [ ]:
categories = {1: "low", 2: "medium", 3: "high"}
data_conf_unordered = (
    data_gdynia["confidence"].map(categories).value_counts().sort_index()
)
data_conf_unordered

In [ ]:
conf_type = pd.CategoricalDtype(categories=["low", "medium", "high"], ordered=True)
data_conf_oredered = data_gdynia["confidence"].map(categories).astype(conf_type)
data_conf_oredered.value_counts().sort_index()

In [ ]:
assert ((data_conf_oredered.value_counts().sort_index()) == [19, 646, 317]).all()
assert (data_conf_oredered >= "medium").sum() == 963
assert data_conf_oredered.max() == "high"

4.

In [ ]:
data_gdynia.assign(
    confidence_level=lambda d: d["confidence"].map(categories).astype(conf_type)
).assign(is_high=lambda d: d["confidence_level"] == "high").groupby("arch_style").agg(
    num_buildings=("osm_id", "size"), share_high=("is_high", "mean")
)


5.

In [ ]:
arch_style_addr_street_unstack_obs = (
    data_gdynia.astype({"arch_style": "category"})
    .loc[data_gdynia["addr_street"].isin(["Świętojańska", "Portowa", "Armii Krajowej"])]
    .groupby(["addr_street", "arch_style"], observed=False)
    .size()
    .unstack()
)
arch_style_addr_street_unstack_obs

In [ ]:
arch_style_addr_street_unstack = (
    data_gdynia.astype({"arch_style": "category"})
    .loc[lambda d: d["addr_street"].isin(["Świętojańska", "Portowa", "Armii Krajowej"])]
    .groupby(["addr_street", "arch_style"])
    .size()
    .unstack()
)
arch_style_addr_street_unstack

In [ ]:
arch_style_addr_street_unstack.sum(axis=1)

In [ ]:
assert (arch_style_addr_street_unstack_obs == 0).sum().sum() == 7
assert arch_style_addr_street_unstack.sum(axis=1).to_dict() == {
    "Armii Krajowej": 25,
    "Portowa": 13,
    "Świętojańska": 106,
}